In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import optuna

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option('display.max_columns', 50)

# --- КОНФІГУРАЦІЯ ---
DATA_FILE_PATH = 'data.csv'
RANDOM_STATE = 42

# --- ЗАВАНТАЖЕННЯ ДАНИХ ---
try:
    data = pd.read_csv(DATA_FILE_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Файл '{DATA_FILE_PATH}' не знайдено.")

# Видалимо непотрібні стовпці
useless_cols = ['Unnamed: 0', 'doctorid', 'country_code']
data.drop(columns=[c for c in useless_cols if c in data.columns], inplace=True)

# --- ЦІЛЬОВА ЗМІННА ---
stay_map = {
    '0-10': 5, '11-20': 15.5, '21-30': 25.5, '31-40': 35.5,
    '41-50': 45.5, '51-60': 55.5, '61-70': 65.5, '71-80': 75.5,
    '81-90': 85.5, '91-100': 95.5, 'More than 100 Days': 105
}
data['Stay_Days'] = data['Stay'].map(stay_map)
data['Stay_Class'] = data['Stay']

# --- ВИЗНАЧЕННЯ ТИПІВ ОЗНАК ---
numerical_features = ['Available Extra Rooms in Hospital', 'Visitors with Patient', 'Admission_Deposit', 'Stay_Days']
categorical_features = [c for c in data.columns if c not in numerical_features + ['case_id', 'patientid', 'Stay', 'Stay_Class']]
for col in categorical_features:
    data[col] = data[col].astype(str)

# --- ФУНКЦІЇ ДЛЯ EDA ---
def plot_histograms(df, features, save_path="histograms.png"):
    n_cols = 2
    n_rows = math.ceil(len(features) / n_cols)
    plt.figure(figsize=(15, n_rows*5))
    for i, col in enumerate(features):
        plt.subplot(n_rows, n_cols, i+1)
        sns.histplot(df[col], bins=50, kde=True)
        plt.title(col)
    plt.tight_layout()
    plt.savefig(save_path)

def plot_boxplots(df, features, save_path="boxplots.png"):
    plt.figure(figsize=(15, 7))
    sns.boxplot(data=df[features], orient='h')
    plt.title('Boxplots')
    plt.savefig(save_path)

def plot_bar_charts(df, features, save_path="bar_charts.png"):
    n_cols = 2
    n_rows = math.ceil(len(features)/n_cols)
    plt.figure(figsize=(15, n_rows*6))
    for i, col in enumerate(features):
        plt.subplot(n_rows, n_cols, i+1)
        counts = df[col].value_counts().nlargest(20)
        sns.barplot(x=counts.values, y=counts.index)
        plt.title(col)
    plt.tight_layout()
    plt.savefig(save_path)

def plot_grouped_boxplots(df, num_feat, cat_features, save_path="grouped_boxplots.png"):
    plt.figure(figsize=(15,10))
    for i, col in enumerate(cat_features[:4]):
        plt.subplot(2, 2, i+1)
        sns.boxplot(x=col, y=num_feat, data=df)
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(save_path)

# --- ОБРОБКА ВИКИДІВ ---
def clip_outliers(df, features):
    df_clean = df.copy()
    for col in features:
        Q1, Q3 = df[col].quantile([0.25,0.75])
        IQR = Q3-Q1
        lower, upper = Q1-1.5*IQR, Q3+1.5*IQR
        df_clean[col] = np.clip(df_clean[col], lower, upper)
    return df_clean

features_to_clip = ['Available Extra Rooms in Hospital', 'Visitors with Patient', 'Admission_Deposit']
data_clean = clip_outliers(data, features_to_clip)

# --- РОЗДІЛЕННЯ НА X/Y ---
ID_cols = ['case_id', 'patientid']
Target_cols = ['Stay', 'Stay_Days', 'Stay_Class']

X_clean = data_clean.drop(columns=ID_cols + Target_cols)
y_clean_reg = data_clean['Stay_Days']
y_clean_clf = data_clean['Stay_Class']

numerical_cols = X_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_clean.select_dtypes(include='object').columns.tolist()

# --- ПАЙПЛАЙН ПРЕПРОЦЕСИНГУ ---
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# --- ФУНКЦІЯ ДЛЯ ТРЕНУВАННЯ МОДЕЛІ ---
def train_model(X, y, model_type='regression'):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    
    if model_type=='regression':
        model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    if model_type=='regression':
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"R2: {r2:.4f}, RMSE: {rmse:.4f}")
    else:
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        print(f"Accuracy: {acc:.4f}, Weighted F1: {f1:.4f}")
    
    return pipeline, X_test, y_test, y_pred

# --- ТРЕНУВАННЯ РЕГРЕСІЇ ---
reg_pipeline, X_test_reg, y_test_reg, y_pred_reg = train_model(X_clean, y_clean_reg, 'regression')
joblib.dump(reg_pipeline, 'regression_model.joblib')

# --- ТРЕНУВАННЯ КЛАСИФІКАЦІЇ ---
clf_pipeline, X_test_clf, y_test_clf, y_pred_clf = train_model(X_clean, y_clean_clf, 'classification')
joblib.dump(clf_pipeline, 'classification_model.joblib')

# --- OPTUNA ДЛЯ РЕГРЕСІЇ ---
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators',50,200),
        'max_depth': trial.suggest_int('max_depth',5,20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf',1,10)
    }
    model = RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1)
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    X_tr, X_val, y_tr, y_val = train_test_split(X_clean, y_clean_reg, test_size=0.2, random_state=RANDOM_STATE)
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, y_pred))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15, show_progress_bar=True)
best_params = study.best_params
print("Optuna Best Params:", best_params)

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1))
])
final_model.fit(X_clean, y_clean_reg)
joblib.dump(final_model, 'final_regression_model.joblib')
print("Фінальна модель регресії збережена.")


R2: 0.4614, RMSE: 15.7016
Accuracy: 0.3942, Weighted F1: 0.3413


[I 2025-11-12 19:50:14,913] A new study created in memory with name: no-name-fe2b5b6a-a525-4c75-97ef-d0cd2a12ac2e


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2025-11-12 19:55:45,504] Trial 0 finished with value: 15.503174451522842 and parameters: {'n_estimators': 124, 'max_depth': 18, 'min_samples_leaf': 9}. Best is trial 0 with value: 15.503174451522842.
[I 2025-11-12 20:02:26,494] Trial 1 finished with value: 15.534572245219222 and parameters: {'n_estimators': 168, 'max_depth': 14, 'min_samples_leaf': 1}. Best is trial 0 with value: 15.503174451522842.
[I 2025-11-12 20:06:29,394] Trial 2 finished with value: 16.105742337988648 and parameters: {'n_estimators': 173, 'max_depth': 6, 'min_samples_leaf': 2}. Best is trial 0 with value: 15.503174451522842.
[I 2025-11-12 20:11:21,585] Trial 3 finished with value: 15.964501810864624 and parameters: {'n_estimators': 181, 'max_depth': 7, 'min_samples_leaf': 7}. Best is trial 0 with value: 15.503174451522842.
[W 2025-11-12 20:12:27,294] Trial 4 failed with parameters: {'n_estimators': 118, 'max_depth': 17, 'min_samples_leaf': 7} because of the following error: KeyboardInterrupt().
Traceback (most

KeyboardInterrupt: 